## Advanced Evaluation – metryki i analiza (multi-label)

### 📋 Cel notebooka
Ewaluacja modelu BERT (multi-label) na zbiorze walidacyjnym:
1. **Metryki medyczne** – Sensitivity, Specificity, PPV, NPV per klasa (z przedziałami ufności)
2. **Metryki z papera (bloodai)** – Accuracy (subset), P/R/F1 (macro/weighted), Hamming, ROC AUC, AUPRC, ECE + wykresy (ROC, Precision–Recall, macierz pomyłek, kalibracja)
3. **Analiza błędów** – FP/FN per klasa, najczęstsze pomyłki, macierz współwystępowań

### 🔄 Wymagania
- `bert_triage_model_*` – wytrenowany model (notebook 05)
- `corpus_val_*.txt` – **TYLKO** zbiór walidacyjny z notebooka 05

### ⚠️ Data Leakage Prevention
- Używamy **tylko** zbioru walidacyjnego (20% corpus, stratified) – bez nakładania z treningiem

### 📌 Multi-label: co oznaczają metryki i macierze

Model przewiduje **wiele klas na raz** (multi-hot). Stąd:

- **Subset accuracy** = odsetek próbek, gdzie **cały zbiór** przewidzianych klas = zbiór prawdziwych klas (najostrzejsza metryka).
- **Precision / Recall / F1 (macro/weighted)** = liczone **per etykieta** (każda klasa osobno), potem uśrednione – poprawne dla multi-label.
- **Macierz pomyłek (co-occurrence)** – `cm[i,j]` = liczba próbek, gdzie **prawda** ma klasę `i` **i** **predykcja** ma klasę `j`. Jedna próbka może wchodzić do wielu komórek. Diagonala = zgoda; poza diagonalą = pomyłki (FN dla i, FP dla j). Wartościowa dla multi-label.
- W tym notebooku używamy **subset accuracy** i **macierzy współwystępowań** (co-occurrence) jako głównych metryk multi-label.

### 🔄 Pipeline: od modelu do wykresów

**1. Wyjście modelu**  
BERT (multi-label) zwraca **logity** – po jednym na klasę, niezależnie. Na wyjściu stosujemy **sigmoid**, nie softmax (każda klasa osobno), więc dostajemy wektor prawdopodobieństw w [0, 1]:
```text
logity (N, 8)  →  sigmoid  →  all_probs (N, 8)
```
Dla jednej próbki: 8 liczb, np. [0.1, 0.9, 0.2, …] = niskie p(POZ), wysokie p(GASTROLOG), niskie p(HEMATOLOG), …

**2. Progi (thresholds)**  
Żeby z prawdopodobieństw zrobić **decyzję binarną** (klasa „tak” / „nie”) dla każdej etykiety, używamy **progu** per klasa:
- **Skąd progi:** W notebooku 05 na zbiorze walidacyjnym wyznaczane są **optimal thresholds** (np. maksymalizacja indeksu Youdena lub F1) i zapisywane w `optimal_thresholds.json` w katalogu modelu. Notebook 07 ładuje je jako `optimal_thresholds_ids` (słownik: id klasy → próg).
- **Jak używamy:** Dla klasy `i`: jeśli `all_probs[sample, i] >= threshold[i]` → przewidujemy „obecna”, w przeciwnym razie „nieobecna”. Daje to wektor multi-hot przewidywań:
```text
y_pred_bin[sample, i] = 1  gdy  all_probs[sample, i] >= threshold[i],  inaczej 0
```
Bez pliku z progami używamy domyślnie 0.5 dla wszystkich klas.

**3. Do czego służy co**  
- **Wykresy kalibracji:** Używamy **surowego** `all_probs` (bez progów). Sprawdzamy, czy gdy model daje np. 0.7, to w ~70% przypadków klasa jest obecna (ECE, Brier, krzywa reliability).
- **Macierze pomyłek 2×2 per klasa i metryki (Precision/Recall itd.):** Używamy **zbinaryzowanych** przewidywań `y_pred_bin` (z progów). Porównujemy z prawdziwymi etykietami binarnymi `y_true_bin` (multi-hot z danych) i liczymy TN, FP, FN, TP per klasa.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.calibration import calibration_curve
from torch.utils.data import Dataset, DataLoader
from transformers import BertForSequenceClassification, PreTrainedTokenizerFast
import torch.nn.functional as F
from tqdm.auto import tqdm
import scipy.stats as stats
from scipy.stats import binom

# --- KONFIGURACJA ---
import paths_config
paths_config.ensure_dirs()
from pathlib import Path
import glob
import os
import sys

# Znajdź najnowszy model fine-tuned (katalog models/triage/)
triage_models = sorted([d for d in glob.glob(str(paths_config.MODELS_TRIAGE / "bert_triage_model_*")) if Path(d).is_dir()], reverse=True)
if not triage_models:
    raise FileNotFoundError("Nie znaleziono modelu bert_triage_model_* w models/triage/. Uruchom najpierw notebook 05.")
MODEL_PATH = triage_models[0]
print(f"📂 Znaleziono model: {MODEL_PATH}")

# Zbiór walidacyjny z notebooka 05 (data/processed/)
val_split_files = sorted(glob.glob(str(paths_config.DATA_PROCESSED / "corpus_val_*.txt")), reverse=True)
if not val_split_files:
    raise FileNotFoundError("Nie znaleziono corpus_val_*.txt w data/processed/. Uruchom najpierw notebook 05.")
DATA_FILE = val_split_files[0]
print(f"📂 Używam zbioru walidacyjnego: {DATA_FILE}")
print(f"⚠️  UWAGA: To jest TYLKO val split z notebooka 05 (nie cały corpus!)")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64

# Mapowanie klas (wspólny moduł dla wszystkich notebooków)

# Znajdź główny katalog projektu (gdzie jest label_mapping.py)
# Szukamy w bieżącym katalogu i katalogach nadrzędnych
current_dir = Path(os.getcwd())
project_root = None

# Sprawdź czy jesteśmy w głównym katalogu projektu
if (current_dir / "label_mapping.py").exists():
    project_root = current_dir
elif (current_dir / "notebooks_ethos" / "label_mapping.py").exists():
    # Jeśli jesteśmy w notebooks_ethos, idź w górę
    project_root = current_dir.parent
else:
    # Szukaj w katalogach nadrzędnych
    for parent in current_dir.parents:
        if (parent / "label_mapping.py").exists():
            project_root = parent
            break

if project_root is None:
    raise FileNotFoundError(
        f"Nie znaleziono pliku label_mapping.py. "
        f"Szukano w: {current_dir} i katalogach nadrzędnych."
    )

# Dodaj główny katalog projektu do sys.path
sys.path.insert(0, str(project_root))
print(f"📂 Dodano do sys.path: {project_root}")

from label_mapping import (
    TARGET_NAMES,  # Short class names (for visualisations)
    LABEL_TO_ID,  # Pełna nazwa -> ID
    ID_TO_LABEL,  # ID -> pełna nazwa
    ID_TO_LABEL_SHORT,  # ID -> krótka nazwa
    NUM_LABELS
)
LABEL_MAP = LABEL_TO_ID  # Dla kompatybilności z istniejącym kodem

print(f"🚀 Device: {DEVICE}")
print(f"📂 Model: {MODEL_PATH}")

# --- Załaduj optimal thresholds z notebooka 05 ---
import json
thresholds_file = Path(MODEL_PATH) / "optimal_thresholds.json"
if thresholds_file.exists():
    with open(thresholds_file, "r") as f:
        optimal_thresholds = json.load(f)
    # Konwertuj z nazw klas (krótkich: 'POZ', 'GASTROLOG' itd.) na ID
    # optimal_thresholds.json używa krótkich nazw, więc musimy użyć ID_TO_LABEL_SHORT
    optimal_thresholds_ids = {}
    for class_name, threshold in optimal_thresholds.items():
        # Szukamy ID dla danej krótkiej nazwy klasy
        for class_id, short_name in ID_TO_LABEL_SHORT.items():
            if short_name == class_name:
                optimal_thresholds_ids[class_id] = threshold
                break
    print(f"✅ Załadowano optimal thresholds z notebooka 05: {optimal_thresholds}")
    print(f"   Thresholds (ID): {optimal_thresholds_ids}")
    if len(optimal_thresholds_ids) == 0:
        print("   ⚠️ Uwaga: Nie udało się zmapować thresholdów na ID - sprawdź format pliku")
else:
    optimal_thresholds = None
    optimal_thresholds_ids = None
    print("⚠️ Nie znaleziono optimal_thresholds.json - użyj domyślnych progów")

plt.savefig(paths_config.OUTPUTS_PLOTS / '07_Advanced_Evaluation_cell_1.png', bbox_inches='tight')


In [ ]:

# --- 1. DATASET & LOADING (MULTI-LABEL) ---
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm.auto import tqdm
from label_mapping import LABEL_MAP, NUM_LABELS, TARGET_NAMES
TARGET_NAMES_EN = [
    'Primary care', 'Gastroenterology', 'Haematology', 'Nephrology',
    'Emergency/ICU', 'Cardiology', 'Pulmonology', 'Hepatology'
]
import torch
from collections import Counter

class EthosEvalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # Multi-label expects FloatTensor for BCEWithLogitsLoss logic
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)

print("📂 Wczytywanie zbioru walidacyjnego (Multi-Label)...")
texts = []
labels = []

# Używamy całego zbioru walidacyjnego
label_keys = set(LABEL_MAP.keys())

valid_lines = 0
skipped_lines = 0

with open(DATA_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()
    print(f"   Całkowita liczba linii w val split: {len(lines)}")
    
    for line in lines:
        line = line.strip()
        if not line: continue
        parts = line.split()
        
        # Parsowanie Multi-Label: Szukamy WSZYSTKICH etykiet w linii
        line_labels = [p for p in parts if p in label_keys]
        line_text_parts = [p for p in parts if p not in label_keys]
        
        if not line_labels:
            skipped_lines += 1
            if skipped_lines < 5:
                print(f"   ⚠️ Pominięto linię (brak etykiet): {line[:50]}...")
            continue
            
        text = " ".join(line_text_parts)
        
        # Tworzymy wektor Multi-Hot (np. [0, 1, 0, 1, 0...])
        label_vec = np.zeros(NUM_LABELS, dtype=np.float32)
        for l in line_labels:
            if l in LABEL_MAP:
                label_vec[LABEL_MAP[l]] = 1.0
            
        texts.append(text)
        labels.append(label_vec)
        valid_lines += 1

print(f"✅ Zbiór walidacyjny gotowy: {len(texts)} próbek (pominięto {skipped_lines}).")

# Walidacja reprezentacji klas (liczymy wystąpienia każdej klasy)
labels_array = np.array(labels)
label_counts = np.sum(labels_array, axis=0) # Suma po kolumnach
missing_classes = []

print("📊 Rozkład klas w zbiorze walidacyjnym:")
for i, count in enumerate(label_counts):
    class_name = TARGET_NAMES[i] if i < len(TARGET_NAMES) else f"ID_{i}"
    print(f"   - {class_name}: {int(count)}")
    if count == 0:
        missing_classes.append(class_name)

if missing_classes:
    raise ValueError(f"❌ KRYTYCZNY BŁĄD: Brakujące klasy w val split: {missing_classes}")

print(f"✅ Wszystkie {NUM_LABELS} klas są reprezentowane.")

# 2. Tokenizer i DataLoader
from pathlib import Path
from tokenizers import Tokenizer

tokenizer_path = Path(MODEL_PATH) / "tokenizer.json"
print(f"🔄 Próba ładowania tokenizera z: {tokenizer_path}")

try:
    # Próba 1: Standardowe ładowanie (często pada na local_files_only jeśli config jest dziwny)
    tokenizer = PreTrainedTokenizerFast.from_pretrained(
        MODEL_PATH,
        local_files_only=True
    )
    print("✅ Tokenizer załadowany (metoda from_pretrained)")
except Exception as e:
    print(f"⚠️ Błąd standardowego ładowania: {e}")
    
    if tokenizer_path.exists():
        print("🔄 Próba 2: Bezpośrednie ładowanie z tokenizer.json...")
        tokenizer_obj = Tokenizer.from_file(str(tokenizer_path))
        tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer_obj)
        # Musimy ręcznie ustawić pad_token, jeśli nie został wczytany
        if tokenizer.pad_token is None:
            tokenizer.pad_token = "[PAD]"
        print("✅ Tokenizer załadowany (metoda alternatywna)")
    else:
        raise FileNotFoundError(f"❌ Nie znaleziono tokenizer.json w {MODEL_PATH}")

test_dataset = EthosEvalDataset(texts, labels, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# --- 3. MODEL LOADING ---
from transformers import BertForSequenceClassification

print(f"📥 Ładowanie modelu z: {MODEL_PATH}")
model = BertForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    local_files_only=True
).to(DEVICE)
model.eval()
print("✅ Model załadowany!")


## 📊 Metryki Medyczne (Clinical Metrics)

### Cel tej sekcji
Obliczenie szczegółowych metryk medycznych wymaganych dla publikacji naukowej:
- **Sensitivity (Recall) per class** - kluczowe dla wykrywania rzadkich przypadków
- **Specificity per class** - ważne dla unikania false positives
- **PPV (Positive Predictive Value)** - co lekarz faktycznie widzi
- **NPV (Negative Predictive Value)** - co model odrzuca

### Uzasadnienie medtech
W medycynie różne metryki mają różne znaczenie:
- **Sensitivity (Recall)**: Najważniejsze dla wykrywania rzadkich, ale krytycznych przypadków (np. HEMATOLOG)
- **Specificity**: Ważne dla unikania niepotrzebnych skierowań (false positives)
- **PPV**: Pokazuje, ile z przewidzianych przypadków jest rzeczywiście prawdziwych (perspektywa lekarza)
- **NPV**: Pokazuje, ile z odrzuconych przypadków jest rzeczywiście negatywnych (perspektywa pacjenta)

In [ ]:
# --- METRYKI MEDYCZNE (CLINICAL METRICS) ---
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import pandas as pd

print("=" * 80)
print("📊 METRYKI MEDYCZNE (CLINICAL METRICS)")
print("=" * 80)

# ⚠️ MULTI-LABEL: all_labels to (n, NUM_LABELS) multi-hot; predykcje z sigmoid + progi z notebooka 05
if 'all_labels' not in globals() or 'y_pred_multi' not in globals():
    print("🔄 Generowanie predykcji dla metryk medycznych (multi-label)...")
    
    model = BertForSequenceClassification.from_pretrained(
        MODEL_PATH,
        num_labels=NUM_LABELS,
        problem_type="multi_label_classification",
        local_files_only=True
    ).to(DEVICE)
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predykcja dla metryk"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels']

            outputs = model(input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # Predykcja multi-hot: progi z notebooka 05 (optimal_thresholds_ids)
    thresh_ids = optimal_thresholds_ids if optimal_thresholds_ids is not None else {}
    thresh = np.array([thresh_ids.get(i, 0.5) for i in range(NUM_LABELS)], dtype=np.float32)
    y_pred_multi = (all_probs >= thresh).astype(np.float32)
    print("✅ Predykcje wygenerowane\n")

# Metryki medyczne per class dla MULTI-LABEL (per-label TP/FP/FN/TN)
def metrics_from_tp_fp_fn_tn(tp, fp, fn, tn, class_name, n_total):
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    prevalence = (tp + fn) / n_total if n_total > 0 else 0.0
    return {
        'Class': class_name,
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'Sensitivity (Recall)': sensitivity,
        'Specificity': specificity,
        'PPV (Precision)': ppv,
        'NPV': npv,
        'Prevalence': prevalence
    }

n_total = all_labels.shape[0]
clinical_metrics = []
for i in range(NUM_LABELS):
    y_true_c = (all_labels[:, i] >= 0.5).astype(np.int32)
    y_pred_c = (y_pred_multi[:, i] >= 0.5).astype(np.int32)
    tp = int(((y_true_c == 1) & (y_pred_c == 1)).sum())
    fp = int(((y_true_c == 0) & (y_pred_c == 1)).sum())
    fn = int(((y_true_c == 1) & (y_pred_c == 0)).sum())
    tn = int(((y_true_c == 0) & (y_pred_c == 0)).sum())
    metrics = metrics_from_tp_fp_fn_tn(tp, fp, fn, tn, TARGET_NAMES[i], n_total)
    clinical_metrics.append(metrics)

# Konwertuj na DataFrame dla łatwego wyświetlania
df_metrics = pd.DataFrame(clinical_metrics)

# Formatowanie dla lepszej czytelności
print("\n📋 TABELA METRYK MEDYCZNYCH PER CLASS:")
print("=" * 80)
print(df_metrics.to_string(index=False, float_format=lambda x: f'{x:.3f}' if isinstance(x, float) else str(x)))

# --- PRZEDZIAŁY UFNOŚCI (95% CI) dla metryk ---
from scipy.stats import binom
import scipy.stats as stats

def calculate_confidence_interval(successes, total, confidence=0.95):
    """
    Oblicza przedział ufności dla proporcji (Wilson score interval).
    
    Args:
        successes: number of successes
        total: Całkowita liczba prób
        confidence: Poziom ufności (domyślnie 0.95)
    
    Returns:
        tuple: (lower_bound, upper_bound)
    """
    if total == 0:
        return (0.0, 0.0)
    
    z = stats.norm.ppf((1 + confidence) / 2)  # 1.96 dla 95% CI
    p = successes / total
    
    # Wilson score interval (lepszy dla małych próbek)
    denominator = 1 + (z**2 / total)
    centre_adjusted_probability = (p + (z**2 / (2 * total))) / denominator
    adjusted_standard_deviation = np.sqrt((p * (1 - p) + z**2 / (4 * total)) / total) / denominator
    
    lower_bound = centre_adjusted_probability - z * adjusted_standard_deviation
    upper_bound = centre_adjusted_probability + z * adjusted_standard_deviation
    
    # Clamp do [0, 1]
    lower_bound = max(0.0, min(1.0, lower_bound))
    upper_bound = max(0.0, min(1.0, upper_bound))
    
    return (lower_bound, upper_bound)

# Dodaj przedziały ufności do metryk (clinical_metrics już obliczone per-class dla multi-label)
clinical_metrics_with_ci = []
for metrics in clinical_metrics:
    # CI dla Sensitivity
    tp = metrics['TP']
    fn = metrics['FN']
    sens_ci = calculate_confidence_interval(tp, tp + fn) if (tp + fn) > 0 else (0.0, 0.0)
    metrics['Sensitivity_CI_lower'] = sens_ci[0]
    metrics['Sensitivity_CI_upper'] = sens_ci[1]
    
    # CI dla Specificity
    tn = metrics['TN']
    fp = metrics['FP']
    spec_ci = calculate_confidence_interval(tn, tn + fp) if (tn + fp) > 0 else (0.0, 0.0)
    metrics['Specificity_CI_lower'] = spec_ci[0]
    metrics['Specificity_CI_upper'] = spec_ci[1]
    
    # CI dla PPV
    ppv_ci = calculate_confidence_interval(tp, tp + fp) if (tp + fp) > 0 else (0.0, 0.0)
    metrics['PPV_CI_lower'] = ppv_ci[0]
    metrics['PPV_CI_upper'] = ppv_ci[1]
    
    # CI dla NPV
    npv_ci = calculate_confidence_interval(tn, tn + fn) if (tn + fn) > 0 else (0.0, 0.0)
    metrics['NPV_CI_lower'] = npv_ci[0]
    metrics['NPV_CI_upper'] = npv_ci[1]
    
    clinical_metrics_with_ci.append(metrics)

# Konwertuj na DataFrame
df_metrics_ci = pd.DataFrame(clinical_metrics_with_ci)

# Formatowanie dla wyświetlania z CI
print("\n📋 TABELA METRYK MEDYCZNYCH Z PRZEDZIAŁAMI UFNOŚCI (95% CI):")
print("=" * 120)
for idx, row in df_metrics_ci.iterrows():
    print(f"\n{row['Class']}:")
    print(f"  Sensitivity: {row['Sensitivity (Recall)']:.3f} [{row['Sensitivity_CI_lower']:.3f}, {row['Sensitivity_CI_upper']:.3f}]")
    print(f"  Specificity: {row['Specificity']:.3f} [{row['Specificity_CI_lower']:.3f}, {row['Specificity_CI_upper']:.3f}]")
    print(f"  PPV:         {row['PPV (Precision)']:.3f} [{row['PPV_CI_lower']:.3f}, {row['PPV_CI_upper']:.3f}]")
    print(f"  NPV:         {row['NPV']:.3f} [{row['NPV_CI_lower']:.3f}, {row['NPV_CI_upper']:.3f}]")

# Zapis do pliku CSV (dla publikacji)
metrics_file = Path(MODEL_PATH) / "clinical_metrics.csv"
df_metrics_ci.to_csv(metrics_file, index=False)
print(f"\n💾 Zapisano metryki medyczne z CI do: {metrics_file}")

# Podsumowanie dla krytycznych klas
print("\n" + "=" * 80)
print("🎯 PODSUMOWANIE DLA KRYTYCZNYCH KLAS:")
print("=" * 80)
critical_classes = ['HEMATOLOG', 'SOR']
for class_name in critical_classes:
    if class_name in df_metrics['Class'].values:
        row = df_metrics[df_metrics['Class'] == class_name].iloc[0]
        print(f"\n{class_name}:")
        print(f"  Sensitivity (Recall): {row['Sensitivity (Recall)']:.3f} - "
              f"{'✅ DOBRE' if row['Sensitivity (Recall)'] > 0.5 else '⚠️ NISKIE' if row['Sensitivity (Recall)'] > 0.3 else '❌ KRYTYCZNE'} "
              f"(wykrywa {row['Sensitivity (Recall)']*100:.1f}% przypadków)")
        print(f"  Specificity: {row['Specificity']:.3f} - "
              f"{'✅ DOBRE' if row['Specificity'] > 0.9 else '⚠️ ŚREDNIE' if row['Specificity'] > 0.7 else '❌ NISKIE'} "
              f"(odrzuca {row['Specificity']*100:.1f}% nieprawdziwych przypadków)")
        print(f"  PPV (Precision): {row['PPV (Precision)']:.3f} - "
              f"{'✅ DOBRE' if row['PPV (Precision)'] > 0.7 else '⚠️ ŚREDNIE' if row['PPV (Precision)'] > 0.5 else '❌ NISKIE'} "
              f"(z przewidzianych {row['PPV (Precision)']*100:.1f}% jest prawdziwych)")
        print(f"  NPV: {row['NPV']:.3f} - "
              f"{'✅ DOBRE' if row['NPV'] > 0.9 else '⚠️ ŚREDNIE' if row['NPV'] > 0.7 else '❌ NISKIE'} "
              f"(z odrzuconych {row['NPV']*100:.1f}% jest rzeczywiście negatywnych)")

print("\n" + "=" * 80)

## Metryki zgodne z paperem (bloodai)

**Źródło:** PDF „Routing kliniczny przy ograniczonej historii pacjenta” – sekcja **IV.A Metryki i protokół**.

Paper wymienia:
- **Accuracy**, **Precision**, **Recall**, **F1** (macro i weighted)
- Dla multi-label: **Hamming Loss**, **ROC AUC per class**, **AUPRC** (Area Under Precision–Recall Curve), **ECE** (Expected Calibration Error)

**Discriminatory power in multi-label:** In multi-label classification there is no single global ROC curve. We measure discriminatory power **per label**: for each class we treat the task as binary (class present vs absent, one-vs-rest). For class *i* the ROC curve is built from (FPR, TPR) pairs using `all_probs[:, i]` and binary ground-truth labels; **the AUC of this curve** is the discriminatory power for that label. The table reports ROC AUC (mean) and per class; **Fig. 1** shows all 8 ROC curves (one per class).

Poniżej: jedna tabela zbiorcza + wykresy odpowiadające rysunkom 1–4 z papera (ROC, Precision–Recall, macierze pomyłek, krzywe kalibracji ECE).

In [ ]:
# --- METRYKI ZGODNE Z PAPEREM (bloodai) - IV.A Metryki i protokół ---
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    hamming_loss, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, confusion_matrix
)
from sklearn.calibration import calibration_curve

# Upewnij się, że mamy predykcje multi-hot
if 'y_pred_multi' not in globals():
    thresh = np.array([optimal_thresholds_ids.get(i, 0.5) for i in range(NUM_LABELS)], dtype=np.float32) if optimal_thresholds_ids else np.full(NUM_LABELS, 0.5)
    y_pred_multi = (all_probs >= thresh).astype(np.float32)

y_true_bin = (all_labels >= 0.5).astype(int)
y_pred_bin = (y_pred_multi >= 0.5).astype(int)

# 1. Accuracy (subset accuracy dla multi-label)
subset_accuracy = (y_pred_bin == y_true_bin).all(axis=1).mean()

# 2. Precision, Recall, F1 – macro i weighted (multi-label)
precision_macro = precision_score(y_true_bin, y_pred_bin, average='macro', zero_division=0)
precision_weighted = precision_score(y_true_bin, y_pred_bin, average='weighted', zero_division=0)
recall_macro = recall_score(y_true_bin, y_pred_bin, average='macro', zero_division=0)
recall_weighted = recall_score(y_true_bin, y_pred_bin, average='weighted', zero_division=0)
f1_macro = f1_score(y_true_bin, y_pred_bin, average='macro', zero_division=0)
f1_weighted = f1_score(y_true_bin, y_pred_bin, average='weighted', zero_division=0)

# 3. Hamming Loss (multi-label)
hamming = hamming_loss(y_true_bin, y_pred_bin)

# 4. ROC AUC per class
roc_auc_per_class = []
for i in range(NUM_LABELS):
    yi = y_true_bin[:, i]
    if yi.sum() == 0 or yi.sum() == len(yi):
        roc_auc_per_class.append(np.nan)
    else:
        roc_auc_per_class.append(roc_auc_score(yi, all_probs[:, i]))
roc_auc_mean = np.nanmean(roc_auc_per_class)

# 5. AUPRC (Area Under Precision–Recall Curve) per class
auprc_per_class = []
for i in range(NUM_LABELS):
    yi = y_true_bin[:, i]
    if yi.sum() == 0:
        auprc_per_class.append(np.nan)
    else:
        auprc_per_class.append(average_precision_score(yi, all_probs[:, i]))
auprc_mean = np.nanmean(auprc_per_class)

# 6. ECE (Expected Calibration Error) per class
def _ece(y_true, y_pred_proba, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        in_bin = (y_pred_proba > lo) & (y_pred_proba <= hi)
        if in_bin.sum() == 0:
            continue
        acc = y_true[in_bin].mean()
        conf = y_pred_proba[in_bin].mean()
        ece += in_bin.mean() * np.abs(acc - conf)
    return ece

ece_per_class = [_ece(y_true_bin[:, i], all_probs[:, i]) for i in range(NUM_LABELS)]
ece_mean = np.mean(ece_per_class)

# 7. F1 per class (for paper table)
f1_per_class = [f1_score(y_true_bin[:, i], y_pred_bin[:, i], zero_division=0) for i in range(NUM_LABELS)]

# ---- Summary table (paper IV.A) ----
print("=" * 80)
print("METRYKI ZGODNE Z PAPEREM (bloodai) – IV.A Metryki i protokół")
print("=" * 80)
paper_metrics = pd.DataFrame({
    'Metric': [
        'Accuracy (subset)', 'Precision (macro)', 'Precision (weighted)',
        'Recall (macro)', 'Recall (weighted)', 'F1 (macro)', 'F1 (weighted)',
        'Hamming Loss', 'ROC AUC (mean)', 'AUPRC (mean)', 'ECE (mean)'
    ],
    'Value': [
        f'{subset_accuracy:.4f}', f'{precision_macro:.4f}', f'{precision_weighted:.4f}',
        f'{recall_macro:.4f}', f'{recall_weighted:.4f}', f'{f1_macro:.4f}', f'{f1_weighted:.4f}',
        f'{hamming:.4f}', f'{roc_auc_mean:.4f}', f'{auprc_mean:.4f}', f'{ece_mean:.4f}'
    ]
})
print(paper_metrics.to_string(index=False))
print("\nROC AUC per class:", dict(zip(TARGET_NAMES_EN, [f'{x:.3f}' if not np.isnan(x) else 'N/A' for x in roc_auc_per_class])))
print("F1 per class:", dict(zip(TARGET_NAMES_EN, [f'{x:.3f}' for x in f1_per_class])))
print("AUPRC per class:", dict(zip(TARGET_NAMES_EN, [f'{x:.3f}' if not np.isnan(x) else 'N/A' for x in auprc_per_class])))
print("ECE per class:", dict(zip(TARGET_NAMES_EN, [f'{x:.4f}' for x in ece_per_class])))

# ---- Rysunek 1: Krzywe ROC (zdolność dyskryminacyjna) ----
fig1, ax1 = plt.subplots(figsize=(10, 6))
for i in range(NUM_LABELS):
    yi = y_true_bin[:, i]
    if yi.sum() == 0 or yi.sum() == len(yi):
        continue
    fpr, tpr, _ = roc_curve(yi, all_probs[:, i])
    roc_auc = roc_auc_score(yi, all_probs[:, i])
    ax1.plot(fpr, tpr, lw=2, label=f'{TARGET_NAMES_EN[i]} (AUC = {roc_auc:.2f})')
ax1.plot([0, 1], [0, 1], 'k--', lw=2)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC curves (discriminatory power) – multi-label BERT model')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)
plt.tight_layout()
paths_config.OUTPUTS_PLOTS.mkdir(parents=True, exist_ok=True)
plt.savefig(paths_config.OUTPUTS_PLOTS / 'roc_discrimiantory_power.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# ---- Rysunek 2: Krzywe Precision–Recall ----
fig2, ax2 = plt.subplots(figsize=(10, 6))
for i in range(NUM_LABELS):
    yi = y_true_bin[:, i]
    if yi.sum() == 0:
        continue
    prec, rec, _ = precision_recall_curve(yi, all_probs[:, i])
    auprc = average_precision_score(yi, all_probs[:, i])
    ax2.plot(rec, prec, lw=2, label=f'{TARGET_NAMES_EN[i]} (AUPRC = {auprc:.2f})')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision–Recall curves – multi-label BERT model')
ax2.legend(loc='lower left')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(paths_config.OUTPUTS_PLOTS / 'precision-recall-curve.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# ---- Rysunek 3: Macierze pomyłek (co-occurrence true vs pred) ----
cm_paper = np.zeros((NUM_LABELS, NUM_LABELS))
for i in range(NUM_LABELS):
    for j in range(NUM_LABELS):
        cm_paper[i, j] = ((y_true_bin[:, i] == 1) & (y_pred_bin[:, j] == 1)).sum()
cm_paper = cm_paper.astype(int)
fig3, ax3 = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_paper, annot=True, fmt='d', cmap='Blues', xticklabels=TARGET_NAMES_EN, yticklabels=TARGET_NAMES_EN, ax=ax3, cbar_kws={'label': 'Count'})
ax3.set_xlabel('Predicted class')
ax3.set_ylabel('Actual class')
ax3.set_title('Confusion matrices (multi-label) – BERT model')
plt.tight_layout()
plt.show()

# ---- Rysunek 4: Krzywe kalibracji (ECE) ----
fig4, ax4 = plt.subplots(figsize=(10, 10))
ax4.plot([0, 1], [0, 1], 'k:', label='Perfectly calibrated')
for i in range(NUM_LABELS):
    yi = y_true_bin[:, i].ravel()
    prob_pos = all_probs[:, i].ravel()
    if yi.sum() == 0 or yi.sum() == len(yi):
        continue
    frac_pos, mean_pred = calibration_curve(yi, prob_pos, n_bins=10, strategy='quantile')
    ax4.plot(mean_pred, frac_pos, 's-', label=f'{TARGET_NAMES_EN[i]}')
ax4.set_ylabel('Actual fraction positive')
ax4.set_xlabel('Mean predicted probability')
ax4.set_title('Calibration curves (ECE) – multi-label BERT model')
ax4.legend(loc='lower right')
ax4.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(paths_config.OUTPUTS_PLOTS / 'calibration_curves.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# ---- Paper: 3 separate 2x2 confusion matrices (SOR, Haematology, Nephrology) ----
for idx, name in [(4, 'SOR'), (2, 'Haematology'), (3, 'Nephrology')]:
    yi_true = y_true_bin[:, idx]
    yi_pred = y_pred_bin[:, idx]
    tn = ((yi_true == 0) & (yi_pred == 0)).sum()
    fp = ((yi_true == 0) & (yi_pred == 1)).sum()
    fn = ((yi_true == 1) & (yi_pred == 0)).sum()
    tp = ((yi_true == 1) & (yi_pred == 1)).sum()
    cm2 = np.array([[tn, fp], [fn, tp]])
    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues', ax=ax, cbar_kws={'label': 'Count'},
                xticklabels=['Pred Neg', 'Pred Pos'], yticklabels=['True Neg', 'True Pos'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Confusion matrix: {name}')
    plt.tight_layout()
    plt.savefig(paths_config.OUTPUTS_PLOTS / f'confusion_{name}.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
print('Saved: confusion_SOR.png, confusion_Haematology.png, confusion_Nephrology.png')

### 📖 Interpretacja dla multi-label – kiedy model „dobrze przewidział”?

**Co uznajemy za poprawne?**
- **Per etykieta:** Dla każdej klasy osobno – predykcja jest dobra, gdy model trafnie przewiduje **obecność** (TP) i **brak** (TN) tej klasy. Liczą się: Sensitivity (wykrywanie przypadków), Specificity (unikanie fałszywych alarmów), PPV/NPV.
- **Cała próbka (subset):** Próbka jest w pełni poprawna tylko wtedy, gdy **cały wektor** przewidzianych klas (multi-hot) jest identyczny z prawdą – czyli subset accuracy. To najostrzejsza miara.

**Predykcja „3 etykiety zamiast 4” – jak to jest liczone?**  
Model dla każdej próbki zwraca **osobno** dla każdej z 8 klas: prawdopodobieństwo → po progu 0/1. W efekcie liczba przewidzianych etykiet może być dowolna (0–8) i często różni się od liczby prawdziwych (np. prawda: 4 klasy, predykcja: 3). Wtedy: **(1)** **Subset accuracy** – taka próbka jest uznawana za **błędną** (wymagamy identycznego zbioru). **(2)** **Precision / Recall / F1 (macro/weighted)** – liczone **per etykieta**: brakująca klasa daje FN, ewentualna nadmiarowa – FP; pozostałe klasy mogą być TP/TN. Te metryki są wartościowe także przy „3 z 4” – pokazują, jak dobrze model radzi sobie z każdą etykietą z osobna. **(3)** **ROC AUC, AUPRC, ECE** – w ogóle nie zależą od tego, „ile etykiet” przewidzieliśmy w jednej próbce; używają ciągłych prawdopodobieństw lub binarnych etykiet per klasa (one-vs-rest). Podsumowując: wyniki **są wartościowe** – subset celowo karze każdą niezgodność; per-label i ROC/AUPRC/ECE poprawnie opisują jakość multi-label.

**Ocena wyników (krótko):** Niski subset accuracy (~31%) w multi-label jest **oczekiwany** – wymagamy identycznego zbioru 8 etykiet; już „3 z 4” lub „4 z 5” trafionych daje subset = 0. Decydujące są metryki per etykieta i probabilistyczne: **ROC AUC (mean)** ~0,92 i **AUPRC** ~0,88 świadczą o **dobrej sile dyskryminacyjnej**; **ECE** ~0,03 – o **dobrej kalibracji**. Precision/Recall/F1 macro ~0,81–0,84 oznaczają sensowną jakość na poziomie pojedynczych etykiet. W kontekście triażu wielospecjalistycznego taki zestaw metryk jest odpowiedni i wyniki można traktować jako wartościowe.

**Jak czytać wykresy kalibracji (per klasa)?**
- **Idealna linia (przekątna):** np. gdy model daje 80% pewności, to w ~80% takich przypadków klasa powinna być obecna.
- **Krzywa nad linią:** model jest **niepewny** (zaniża prawdopodobieństwa).
- **Krzywa pod linią:** model jest **zbyt pewny** (zawyża prawdopodobieństwa).
- **ECE (Expected Calibration Error):** im mniejszy, tym lepsza kalibracja; Brier – im mniejszy, tym lepsza jakość probabilistyczna.

**Jak czytać macierze pomyłek per klasa (2×2)?**
- Dla każdej specjalności osobno: **Actual** = czy w prawdzie była ta klasa, **Predicted** = czy model ją przewidział (po progu).
- **TN** – dobrze odrzucona; **FP** – fałszywy alarm; **FN** – przeoczenie; **TP** – trafna detekcja.
- W multi-label jedna próbka wchodzi do wielu takich macierzy (po jednej komórce w każdej klasie). Priorytet: wysoka Sensitivity dla klas krytycznych (np. SOR), rozsądna Specificity żeby nie generować zbędnych skierowań.

In [ ]:
# --- WYKRESY PER-LABEL: KALIBRACJA (8×) + CONFUSION MATRIX (8×) ---
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

# Progi (jak w reszcie notebooka)
thresh_list = np.array([optimal_thresholds_ids.get(i, 0.5) for i in range(NUM_LABELS)]) if optimal_thresholds_ids is not None else np.full(NUM_LABELS, 0.5)
y_pred_bin = (all_probs >= thresh_list).astype(int)
y_true_bin = (all_labels >= 0.5).astype(int)

def _ece_bin(y_true, y_pred_proba, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        in_bin = (y_pred_proba > lo) & (y_pred_proba <= hi)
        if in_bin.sum() == 0:
            continue
        acc = y_true[in_bin].mean()
        conf = y_pred_proba[in_bin].mean()
        ece += in_bin.mean() * np.abs(acc - conf)
    return ece

# ---- 1. Kalibracja per klasa (8 subplotów, ECE + Brier w tytule) ----
fig_cal, axes_cal = plt.subplots(2, 4, figsize=(16, 10))
axes_cal = axes_cal.flatten()
for i in range(NUM_LABELS):
    ax = axes_cal[i]
    yi = y_true_bin[:, i].ravel()
    prob_pos = all_probs[:, i].ravel()
    ece = _ece_bin(yi, prob_pos)
    brier = brier_score_loss(yi, prob_pos)
    ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Ideal')
    if yi.sum() > 0 and yi.sum() < len(yi):
        frac_pos, mean_pred = calibration_curve(yi, prob_pos, n_bins=10, strategy='quantile')
        ax.plot(mean_pred, frac_pos, 's-', color='#1f77b4', lw=2, markersize=6)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(f'{TARGET_NAMES_EN[i]}\nECE={ece:.3f}  Brier={brier:.3f}', fontsize=11)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction positive')
    ax.grid(True, alpha=0.3)
fig_cal.suptitle('Calibration per class (multi-label)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ---- 2. Confusion matrix per klasa (8× 2×2, z progiem w tytule) ----
fig_cm, axes_cm = plt.subplots(2, 4, figsize=(16, 10))
axes_cm = axes_cm.flatten()
for i in range(NUM_LABELS):
    ax = axes_cm[i]
    yi_true = y_true_bin[:, i]
    yi_pred = y_pred_bin[:, i]
    tn = ((yi_true == 0) & (yi_pred == 0)).sum()
    fp = ((yi_true == 0) & (yi_pred == 1)).sum()
    fn = ((yi_true == 1) & (yi_pred == 0)).sum()
    tp = ((yi_true == 1) & (yi_pred == 1)).sum()
    cm2 = np.array([[tn, fp], [fn, tp]])
    sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues', ax=ax, cbar_kws={'label': 'Count'},
                xticklabels=['Pred False', 'Pred True'], yticklabels=['Actual False', 'Actual True'])
    ax.set_title(f'{TARGET_NAMES_EN[i]}  (Thr: {thresh_list[i]:.2f})', fontsize=11)
fig_cm.suptitle('Confusion matrix per class (2×2, multi-label)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🔍 Analiza Błędów (Error Analysis)

### Cel tej sekcji
Szczegółowa analiza błędów klasyfikacji:
- **False Positives (FP)**: Przypadki błędnie sklasyfikowane jako dana klasa
- **False Negatives (FN)**: Przypadki danej klasy błędnie sklasyfikowane jako inne
- **Przykłady błędnych klasyfikacji** z analizą przyczyn
- **Wizualizacja błędów** w confusion matrix

### Uzasadnienie medtech
Analiza błędów jest kluczowa dla:
- Zrozumienia ograniczeń modelu
- Identyfikacji przypadków wymagających uwagi klinicznej
- Poprawy modelu w przyszłości

In [ ]:
# --- ANALIZA BŁĘDÓW (ERROR ANALYSIS) — multi-label ---
print("=" * 80)
print("🔍 ANALIZA BŁĘDÓW (ERROR ANALYSIS)")
print("=" * 80)

# Primary pred per sample (dla confusion_pairs i wyświetlania)
y_pred_argmax = np.argmax(all_probs, axis=1)
if 'y_pred_multi' not in globals():
    thresh = optimal_thresholds_ids if optimal_thresholds_ids else {i: 0.5 for i in range(NUM_LABELS)}
    y_pred_multi = (all_probs >= np.array([thresh.get(i, 0.5) for i in range(NUM_LABELS)], dtype=np.float32)).astype(np.float32)

# 1. Identyfikacja błędów per-class (multi-label: FP/FN per klasa)
errors = {
    'false_positives': {},
    'false_negatives': {},
    'confusion_pairs': []  # pary (primary true, primary pred, idx) dla niezgodności
}

for i in range(NUM_LABELS):
    class_name = TARGET_NAMES[i]
    errors['false_positives'][class_name] = []
    errors['false_negatives'][class_name] = []

n_samples = all_labels.shape[0]
for idx in range(n_samples):
    for c in range(NUM_LABELS):
        true_c = 1 if (all_labels[idx, c] >= 0.5) else 0
        pred_c = 1 if (y_pred_multi[idx, c] >= 0.5) else 0
        if true_c == 1 and pred_c == 0:
            errors['false_negatives'][TARGET_NAMES[c]].append(idx)
        if true_c == 0 and pred_c == 1:
            errors['false_positives'][TARGET_NAMES[c]].append(idx)
    # confusion_pairs: primary true vs primary pred (dla statystyk „najczęstsze pomyłki”)
    primary_true = int(np.argmax(all_labels[idx]))
    primary_pred = int(y_pred_argmax[idx])
    if primary_true != primary_pred:
        errors['confusion_pairs'].append((TARGET_NAMES[primary_true], TARGET_NAMES[primary_pred], idx))

# 2. Statystyki błędów
print("\n📊 STATYSTYKI BŁĘDÓW:")
print("=" * 80)
for i in range(NUM_LABELS):
    class_name = TARGET_NAMES[i]
    fp_count = len(errors['false_positives'][class_name])
    fn_count = len(errors['false_negatives'][class_name])
    total_class = int((all_labels[:, i] >= 0.5).sum())
    
    print(f"\n{class_name}:")
    print(f"  False Positives: {fp_count} (błędnie sklasyfikowane jako {class_name})")
    print(f"  False Negatives: {fn_count} (prawdziwe {class_name} błędnie sklasyfikowane)")
    print(f"  Total cases: {total_class}")
    if total_class > 0:
        print(f"  Error rate: {(fp_count + fn_count) / total_class * 100:.2f}%")

# 3. Najczęstsze pomyłki (confusion pairs – primary class)
from collections import Counter
confusion_counter = Counter([(true, pred) for true, pred, _ in errors['confusion_pairs']])
n_pairs = len(errors['confusion_pairs'])
print("\n\n🔴 NAJCZĘSTSZE POMYŁKI (primary class):")
print("=" * 80)
for (true_class, pred_class), count in confusion_counter.most_common(10):
    pct = (count / n_pairs * 100) if n_pairs > 0 else 0.0
    print(f"  {true_class} → {pred_class}: {count} przypadków ({pct:.2f}%)")

# 4. Przykłady błędnych klasyfikacji
print("\n\n📝 PRZYKŁADY BŁĘDNYCH KLASYFIKACJI:")
print("=" * 80)

# Wybierz kilka przykładów dla każdej krytycznej klasy
critical_classes = ['HEMATOLOG', 'SOR']
examples_per_class = 3

for class_name in critical_classes:
    class_idx = TARGET_NAMES.index(class_name) if class_name in TARGET_NAMES else None
    if class_idx is None:
        continue
    
    # False Negatives: prawdziwe przypadki błędnie sklasyfikowane
    fn_indices = errors['false_negatives'][class_name][:examples_per_class]
    
    if len(fn_indices) > 0:
        print(f"\n❌ False Negatives dla {class_name} (prawdziwe {class_name} błędnie sklasyfikowane):")
        for idx in fn_indices:
            true_label = int(np.argmax(all_labels[idx]))
            pred_label = int(y_pred_argmax[idx])
            true_probs = all_probs[idx]
            
            print(f"\n  Przykład {idx}:")
            print(f"    Tekst: {texts[idx][:100]}...")
            true_classes = ", ".join(TARGET_NAMES[c] for c in range(NUM_LABELS) if all_labels[idx, c] >= 0.5)
            pred_classes = ", ".join(TARGET_NAMES[c] for c in range(NUM_LABELS) if y_pred_multi[idx, c] >= 0.5)
            print(f"    Prawdziwe klasy: {true_classes}")
            print(f"    Przewidziane klasy: {pred_classes}")
            print(f"    Probabilities:")
            for j, prob in enumerate(true_probs):
                marker = " ← PRZEWIDZIANA" if j == pred_label else " ← PRAWDZIWA" if j == true_label else ""
                print(f"      {TARGET_NAMES[j]}: {prob:.3f}{marker}")

# 5. Wizualizacja - macierz współwystępowań (multi-label: true class i vs pred class j)
cm = np.zeros((NUM_LABELS, NUM_LABELS))
for i in range(NUM_LABELS):
    for j in range(NUM_LABELS):
        cm[i, j] = ((all_labels[:, i] >= 0.5) & (y_pred_multi[:, j] >= 0.5)).sum()

cm = cm.astype(int)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=TARGET_NAMES_EN, 
            yticklabels=TARGET_NAMES_EN, ax=ax, cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted class', fontsize=12)
ax.set_ylabel('Actual class', fontsize=12)
ax.set_title('Confusion Matrix z adnotacją błędów', fontsize=14, fontweight='bold')

# Dodaj adnotacje dla błędów
for i in range(NUM_LABELS):
    for j in range(NUM_LABELS):
        if i != j and cm[i, j] > 0:
            # To jest błąd (false negative dla klasy i, false positive dla klasy j)
            ax.text(j+0.5, i+0.7, f'FN→{TARGET_NAMES_EN[i]}\nFP→{TARGET_NAMES_EN[j]}', 
                   ha='center', va='center', fontsize=8, color='red', fontweight='bold')

plt.tight_layout()
error_matrix_file = Path(MODEL_PATH) / "error_analysis_confusion_matrix.png"
plt.savefig(error_matrix_file, dpi=300, bbox_inches='tight')
print(f"\n💾 Zapisano confusion matrix z adnotacjami do: {error_matrix_file}")
plt.show()

# 6. Zapis szczegółowej analizy błędów do pliku
error_analysis_file = Path(MODEL_PATH) / "error_analysis.txt"
with open(error_analysis_file, 'w', encoding='utf-8') as f:
    f.write("ANALIZA BŁĘDÓW KLASYFIKACJI\n")
    f.write("=" * 80 + "\n\n")
    
    f.write("STATYSTYKI BŁĘDÓW:\n")
    f.write("-" * 80 + "\n")
    for i in range(NUM_LABELS):
        class_name = TARGET_NAMES[i]
        fp_count = len(errors['false_positives'][class_name])
        fn_count = len(errors['false_negatives'][class_name])
        f.write(f"{class_name}: FP={fp_count}, FN={fn_count}\n")
    
    f.write("\n\nNAJCZĘSTSZE POMYŁKI:\n")
    f.write("-" * 80 + "\n")
    for (true_class, pred_class), count in confusion_counter.most_common(20):
        f.write(f"{true_class} → {pred_class}: {count} przypadków\n")
    
    f.write("\n\nPRZYKŁADY BŁĘDNYCH KLASYFIKACJI:\n")
    f.write("-" * 80 + "\n")
    for class_name in critical_classes:
        fn_indices = errors['false_negatives'][class_name][:10]  # 10 przykładów
        if len(fn_indices) > 0:
            f.write(f"\nFalse Negatives dla {class_name}:\n")
            for idx in fn_indices:
                f.write(f"\n  Indeks {idx}:\n")
                f.write(f"    Tekst: {texts[idx]}\n")
                true_str = ", ".join(TARGET_NAMES[c] for c in range(NUM_LABELS) if all_labels[idx, c] >= 0.5)
                pred_str = ", ".join(TARGET_NAMES[c] for c in range(NUM_LABELS) if y_pred_multi[idx, c] >= 0.5)
                f.write(f"    Prawdziwe: {true_str}\n")
                f.write(f"    Przewidziane: {pred_str}\n")
                f.write(f"    Probabilities: {dict(zip(TARGET_NAMES_EN, all_probs[idx]))}\n")

print(f"💾 Zapisano szczegółową analizę błędów do: {error_analysis_file}")
print("\n" + "=" * 80)